# Автоматическая адаптация англоязычных датасетов под культурные нормы России

**Пайплайн:**
1. Загрузка датасетов (CrowS-Pairs, SNIPS)
2. NER-разметка культурно-специфичных сущностей
3. Метод 1: Наивный машинный перевод (Google Translate)
4. Метод 2: Наивный zero-shot LLM перевод
5. Метод 3: Полный пайплайн адаптации (NER → перевод сущностей → несколько вариантов → LLM-Judge → лучший)
6. Сохранение всех вариантов датасетов
7. Оценка: Bias-метрика (CrowS-Pairs), Slot F1 + Intent Accuracy (SNIPS)
8. Проверка культурного сдвига через LLM
9. Агрегированное сравнение результатов

In [ ]:
import sys
sys.path.insert(0, "D:/adaptation")

import logging
import warnings
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config import PipelineConfig
from src.data_loader import load_crows_pairs, load_snips, snips_to_bio
from src.ner_processor import NERProcessor
from src.mapping_store import MappingStore
from src.llm_client import LLMClient
from src.naive_translator import translate_crows_pairs as naive_translate_crows, translate_snips as naive_translate_snips
from src.naive_llm_translator import translate_crows_pairs as llm_translate_crows, translate_snips as llm_translate_snips
from src.pipeline_adapter import adapt_crows_pairs, adapt_snips
from src.judge import JudgeEvaluator
from src.label_checker import check_crows_pair, check_snips_utterance
from src.bias_evaluator import run_bias_evaluation
from src.slot_evaluator import run_slot_evaluation, build_label_maps
from src.cultural_shift_checker import run_shift_evaluation
from src.visualization import (
    plot_bias_comparison, plot_slot_comparison,
    plot_cultural_shift, build_summary_table, format_summary,
)

print("All modules imported successfully.")

## 1. Конфигурация

Укажите ваш API-ключ OpenRouter и модель.

In [ ]:
cfg = PipelineConfig()
cfg.paths.ensure_dirs()

# ===== ЗАПОЛНИТЕ ВАШИ ДАННЫЕ =====
cfg.openrouter.api_key = ""   # <-- ваш ключ OpenRouter
cfg.openrouter.model = ""     # <-- модель, напр. "anthropic/claude-sonnet-4" или "openai/gpt-4o"
# =================================

print(f"Data root: {cfg.paths.root}")
print(f"Model: {cfg.openrouter.model or 'NOT SET'}")
print(f"Bias eval models: {cfg.evaluation.bias_models}")
print(f"Slot eval models: {cfg.evaluation.slot_models}")
print(f"Runs per evaluation: {cfg.evaluation.n_runs}")

## 2. Загрузка датасетов

In [ ]:
# Загрузка CrowS-Pairs
crows_df = load_crows_pairs(cfg.paths)
print(f"CrowS-Pairs: {len(crows_df)} pairs")
print(f"Bias types: {crows_df['bias_type'].value_counts().to_dict()}")
crows_df.head(3)

In [ ]:
# Загрузка SNIPS
snips_df = load_snips(cfg.paths)
print(f"SNIPS: {len(snips_df)} utterances, {snips_df['intent'].nunique()} intents")
print(f"Intents: {snips_df['intent'].value_counts().to_dict()}")

# Добавим BIO-теги для оригинального датасета
snips_df["bio_tags"] = snips_df.apply(lambda r: snips_to_bio(r.to_dict()), axis=1)
snips_df.head(3)

## 3. NER-разметка культурно-специфичных сущностей

In [ ]:
# Инициализация NER
ner = NERProcessor(cfg.spacy_model)

# Пример NER на CrowS-Pairs
sample = crows_df.iloc[0]
print(f"sent_more: {sample['sent_more']}")
print(f"Entities: {[(e.text, e.label, e.adaptation_type) for e in ner.extract_cultural_entities(sample['sent_more'])]}")

# Пример diff между парой
diff_a, diff_b = ner.diff_entities(sample["sent_more"], sample["sent_less"])
print(f"\nDiffering entities (sent_more only): {[e.text for e in diff_a]}")
print(f"Differing entities (sent_less only): {[e.text for e in diff_b]}")

In [ ]:
# Статистика по культурным сущностям в CrowS-Pairs
crows_df["has_cultural_ents"] = crows_df["sent_more"].apply(ner.has_cultural_entities)
print(f"Пары с культурными сущностями: {crows_df['has_cultural_ents'].sum()} / {len(crows_df)}")
print(f"({crows_df['has_cultural_ents'].mean()*100:.1f}%)")

# Статистика по SNIPS
snips_df["has_cultural_ents"] = snips_df["text"].apply(ner.has_cultural_entities)
print(f"\nSNIPS с культурными сущностями: {snips_df['has_cultural_ents'].sum()} / {len(snips_df)}")
print(f"({snips_df['has_cultural_ents'].mean()*100:.1f}%)")

## 4. Метод 1: Наивный машинный перевод (Google Translate)

Простой перевод без культурной адаптации — сохраняет американские реалии на русском языке.

In [ ]:
# CrowS-Pairs: наивный перевод
crows_naive = naive_translate_crows(crows_df)
crows_naive.to_csv(cfg.paths.data_naive_translated / "crows_pairs_naive.csv", index=False)
print(f"Сохранено: {cfg.paths.data_naive_translated / 'crows_pairs_naive.csv'}")
crows_naive[["sent_more", "sent_more_ru", "sent_less", "sent_less_ru"]].head(3)

In [ ]:
# SNIPS: наивный перевод
snips_naive = naive_translate_snips(snips_df)
snips_naive.to_json(cfg.paths.data_naive_translated / "snips_naive.jsonl", orient="records", lines=True, force_ascii=False)
print(f"Сохранено: {cfg.paths.data_naive_translated / 'snips_naive.jsonl'}")
snips_naive[["text", "text_ru", "intent"]].head(3)

## 5. Метод 2: Наивный zero-shot LLM перевод

LLM переводит с минимальным промптом — без структурированной методологии адаптации.

In [ ]:
# Инициализация LLM-клиента (нужен API key!)
assert cfg.openrouter.api_key, "Установите cfg.openrouter.api_key выше!"
assert cfg.openrouter.model, "Установите cfg.openrouter.model выше!"

llm = LLMClient(cfg.openrouter)
print(f"LLM client ready: {cfg.openrouter.model}")

In [ ]:
# CrowS-Pairs: zero-shot LLM перевод
crows_llm = llm_translate_crows(llm, crows_df)
crows_llm.to_csv(cfg.paths.data_naive_llm / "crows_pairs_llm.csv", index=False)
print(f"Сохранено: {cfg.paths.data_naive_llm / 'crows_pairs_llm.csv'}")
crows_llm[["sent_more", "sent_more_ru", "sent_less", "sent_less_ru"]].head(3)

In [ ]:
# SNIPS: zero-shot LLM перевод
snips_llm = llm_translate_snips(llm, snips_df)
snips_llm.to_json(cfg.paths.data_naive_llm / "snips_llm.jsonl", orient="records", lines=True, force_ascii=False)
print(f"Сохранено: {cfg.paths.data_naive_llm / 'snips_llm.jsonl'}")
snips_llm[["text", "text_ru", "intent"]].head(3)

## 6. Метод 3: Полный пайплайн культурной адаптации

**Шаги:**
1. NER-детекция культурно-специфичных сущностей
2. Генерация N вариантов адаптации через LLM с Anthropological Prompting
3. LLM-as-a-Judge выбирает лучший вариант
4. Проверка сохранения лейбла
5. Обновление базы маппингов для консистентности

In [ ]:
# Инициализация компонентов пайплайна
mapping_store = MappingStore(cfg.paths.data_adapted / "mappings.json")
judge = JudgeEvaluator(llm)

print(f"Mapping store: {len(mapping_store)} existing mappings")
print(f"Variants per example: {cfg.openrouter.n_variants}")

In [ ]:
# CrowS-Pairs: полный пайплайн адаптации
crows_adapted = adapt_crows_pairs(
    llm, ner, mapping_store, judge, crows_df,
    n_variants=cfg.openrouter.n_variants,
)
crows_adapted.to_csv(cfg.paths.data_adapted / "crows_pairs_adapted.csv", index=False)
print(f"Сохранено: {cfg.paths.data_adapted / 'crows_pairs_adapted.csv'}")
print(f"Маппингов в базе: {len(mapping_store)}")
crows_adapted[["sent_more", "sent_more_ru", "sent_less", "sent_less_ru", "bias_type"]].head(3)

In [ ]:
# SNIPS: полный пайплайн адаптации
snips_adapted = adapt_snips(
    llm, mapping_store, judge, snips_df,
    n_variants=cfg.openrouter.n_variants,
)
snips_adapted.to_json(cfg.paths.data_adapted / "snips_adapted.jsonl", orient="records", lines=True, force_ascii=False)
print(f"Сохранено: {cfg.paths.data_adapted / 'snips_adapted.jsonl'}")
print(f"Маппингов в базе: {len(mapping_store)}")
snips_adapted[["text", "text_ru", "intent"]].head(3)

In [ ]:
# Просмотр базы маппингов
import json
mappings = json.loads((cfg.paths.data_adapted / "mappings.json").read_text(encoding="utf-8"))
print(f"Всего маппингов: {len(mappings)}")
# Показать первые 20
for orig, adapted in list(mappings.items())[:20]:
    print(f"  {orig} → {adapted}")

## 7. Проверка сохранения лейблов (выборочная)

Используем LLM для cross-check — корректны ли лейблы после адаптации.

In [ ]:
# Выборочная проверка лейблов CrowS-Pairs (на 30 примерах)
label_check_sample = crows_adapted.sample(n=min(30, len(crows_adapted)), random_state=42)
valid_count = 0
issues = []

for _, row in label_check_sample.iterrows():
    try:
        result = check_crows_pair(
            llm,
            original={"sent_more": row["sent_more"], "sent_less": row["sent_less"],
                       "bias_type": row["bias_type"], "stereo_antistereo": row["stereo_antistereo"]},
            adapted={"sent_more_ru": row["sent_more_ru"], "sent_less_ru": row["sent_less_ru"]},
        )
        if result.get("valid"):
            valid_count += 1
        else:
            issues.append({"index": row.name, "issues": result.get("issues", [])})
    except Exception as e:
        issues.append({"index": row.name, "issues": [str(e)]})

print(f"Валидных пар: {valid_count}/{len(label_check_sample)} ({valid_count/len(label_check_sample)*100:.0f}%)")
if issues:
    print(f"\nПримеры проблем:")
    for iss in issues[:5]:
        print(f"  idx={iss['index']}: {iss['issues']}")

## 8. Оценка: CrowS-Pairs Bias Metric

Pseudo-log-likelihood metric: % пар, где модель предпочитает стереотипное предложение.
Прогоняем на нескольких MLM-моделях × всех методах перевода.

In [ ]:
# Оценка bias на всех вариантах датасета
crows_datasets = {
    "Naive MT": crows_naive,
    "Zero-shot LLM": crows_llm,
    "Pipeline": crows_adapted,
}

bias_results = {}
for method, df in crows_datasets.items():
    print(f"\n{'='*50}")
    print(f"Evaluating bias: {method}")
    print(f"{'='*50}")
    bias_results[method] = run_bias_evaluation(
        df,
        model_names=cfg.evaluation.bias_models,
        sent_more_col="sent_more_ru",
        sent_less_col="sent_less_ru",
        n_runs=cfg.evaluation.n_runs,
    )
    print(bias_results[method].groupby("model")["metric_score"].agg(["mean", "std"]))

## 9. Оценка: SNIPS Slot Filling & Intent Classification

Дообучаем BERT-модели на каждом варианте перевода SNIPS, тестируем на одном тест-сете.

> **Примечание**: для корректной оценки slot filling нужны BIO-теги для переведённых текстов. Для наивного перевода теги берутся из оригинала (позиционное приближение), для pipeline-адаптации — из ответа LLM.

In [ ]:
# Подготовка BIO-тегов для переведённых версий SNIPS
# Для наивного перевода: токены могут не совпадать, используем оригинальные теги с обрезкой
def align_bio_tags_naive(row):
    """Грубое выравнивание BIO-тегов: берём оригинальные, обрезаем/дополняем по длине."""
    if not row.get("text_ru") or pd.isna(row["text_ru"]):
        return ["O"]
    n_tokens_ru = len(str(row["text_ru"]).split())
    orig_tags = row["bio_tags"]
    if n_tokens_ru <= len(orig_tags):
        return orig_tags[:n_tokens_ru]
    return orig_tags + ["O"] * (n_tokens_ru - len(orig_tags))

# Применяем к наивному и LLM-переводу
for df_name, df in [("snips_naive", snips_naive), ("snips_llm", snips_llm)]:
    df["bio_tags_ru"] = df.apply(lambda r: align_bio_tags_naive(r.to_dict()), axis=1)
    print(f"{df_name}: bio_tags_ru added")

# Для pipeline-адаптации: используем slots_ru из ответа LLM
def build_bio_from_slots_ru(row):
    """Построить BIO-теги из slots_ru для адаптированного текста."""
    text = row.get("text_ru", "")
    slots = row.get("slots_ru", [])
    if not text or pd.isna(text) or not slots:
        return ["O"] * max(len(str(text).split()), 1)
    tokens = str(text).split()
    tags = ["O"] * len(tokens)
    for slot in slots:
        if not isinstance(slot, dict):
            continue
        slot_text = slot.get("text", "")
        slot_name = slot.get("slot", "UNK")
        # Простой поиск подстроки
        slot_tokens = slot_text.split()
        for i in range(len(tokens) - len(slot_tokens) + 1):
            if tokens[i:i+len(slot_tokens)] == slot_tokens:
                tags[i] = f"B-{slot_name}"
                for j in range(1, len(slot_tokens)):
                    tags[i+j] = f"I-{slot_name}"
                break
    return tags

snips_adapted["bio_tags_ru"] = snips_adapted.apply(
    lambda r: build_bio_from_slots_ru(r.to_dict()), axis=1
)
print("snips_adapted: bio_tags_ru added")

In [ ]:
# Train/test split (80/20) — один и тот же split для всех методов
from sklearn.model_selection import train_test_split

split_indices = train_test_split(range(len(snips_df)), test_size=0.2, random_state=42, stratify=snips_df["intent"])
train_idx, test_idx = split_indices

snips_methods = {
    "Naive MT": snips_naive,
    "Zero-shot LLM": snips_llm,
    "Pipeline": snips_adapted,
}

slot_results = {}
for method, df in snips_methods.items():
    print(f"\n{'='*50}")
    print(f"Evaluating slots: {method}")
    print(f"{'='*50}")

    # Фильтруем строки с пустым переводом
    df_valid = df[df["text_ru"].notna() & (df["text_ru"] != "")].copy()
    train_df = df_valid.iloc[[i for i in train_idx if i < len(df_valid)]]
    test_df = df_valid.iloc[[i for i in test_idx if i < len(df_valid)]]

    slot_results[method] = run_slot_evaluation(
        train_df, test_df,
        model_names=cfg.evaluation.slot_models,
        text_col="text_ru",
        bio_tags_col="bio_tags_ru",
        n_runs=cfg.evaluation.n_runs,
        epochs=cfg.evaluation.slot_epochs,
        batch_size=cfg.evaluation.slot_batch_size,
        lr=cfg.evaluation.slot_lr,
    )
    print(slot_results[method].groupby("model")[["intent_accuracy", "slot_f1"]].agg(["mean", "std"]))

## 10. Проверка культурного сдвига через LLM

LLM оценивает качество культурной адаптации (1-5) на выборке из каждого метода.

In [ ]:
# Проверка культурного сдвига на CrowS-Pairs
shift_results = run_shift_evaluation(
    llm,
    datasets=crows_datasets,
    sent_col="sent_more_ru",
    sample_size=100,
    n_runs=cfg.evaluation.n_runs,
)

print("Результаты проверки культурного сдвига:")
print(shift_results.groupby("method")[["mean_score"]].agg(["mean", "std"]))

## 11. Агрегированные результаты и визуализация

Сводная таблица и графики по всем метрикам × всем методам × всем моделям.

In [ ]:
# Сводная таблица
summary = build_summary_table(bias_results, slot_results, shift_results)
formatted = format_summary(summary)
print("=" * 80)
print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ (mean ± std по нескольким прогонам)")
print("=" * 80)
formatted

In [ ]:
# График: Bias metric по методам и моделям
fig_bias = plot_bias_comparison(bias_results)
fig_bias.savefig(cfg.paths.root / "bias_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# График: Slot filling / Intent classification по методам и моделям
fig_slot = plot_slot_comparison(slot_results)
fig_slot.savefig(cfg.paths.root / "slot_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# График: Cultural shift quality
fig_shift = plot_cultural_shift(shift_results)
fig_shift.savefig(cfg.paths.root / "cultural_shift.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Детальная таблица bias по типам bias
print("BIAS METRIC ПО ТИПАМ ПРЕДУБЕЖДЕНИЙ")
print("=" * 80)
for method, df in bias_results.items():
    print(f"\n--- {method} ---")
    bias_cols = [c for c in df.columns if c.startswith("bias_")]
    if bias_cols:
        print(df[["model"] + bias_cols].groupby("model").mean().round(3).to_string())

## 12. Сохранение всех результатов

In [ ]:
# Сохранение результатов оценки
results_dir = cfg.paths.root / "results"
results_dir.mkdir(exist_ok=True)

# Bias results
for method, df in bias_results.items():
    df.to_csv(results_dir / f"bias_{method.replace(' ', '_').lower()}.csv", index=False)

# Slot results
for method, df in slot_results.items():
    df.to_csv(results_dir / f"slot_{method.replace(' ', '_').lower()}.csv", index=False)

# Shift results
shift_results.to_csv(results_dir / "cultural_shift.csv", index=False)

# Summary
summary.to_csv(results_dir / "summary.csv", index=False)
formatted.to_csv(results_dir / "summary_formatted.csv", index=False)

print(f"Все результаты сохранены в {results_dir}")
print(f"\nФайлы:")
for f in sorted(results_dir.glob("*.csv")):
    print(f"  {f.name}")

## Сохранённые датасеты

| Метод | CrowS-Pairs | SNIPS |
|-------|-------------|-------|
| Наивный MT | `data/naive_translated/crows_pairs_naive.csv` | `data/naive_translated/snips_naive.jsonl` |
| Zero-shot LLM | `data/naive_llm/crows_pairs_llm.csv` | `data/naive_llm/snips_llm.jsonl` |
| Pipeline | `data/adapted/crows_pairs_adapted.csv` | `data/adapted/snips_adapted.jsonl` |
| Маппинги | — | `data/adapted/mappings.json` |